# v9 Clean Direct SFT

## 概要
- **データセット**: `inputs/cleaned/merged_dataset_final_clean.jsonl`（Person Yのクリーニング済み）
- **特徴**: CoT除去済み、構文検証済み、1500件
- **アプローチ**: Empty Think Injection なしでまず試す

## v8からの改善点
- 学習データから「Approach:」「Output:」が完全に削除されている
- 構文検証済みの高品質データのみ使用

## 0. 環境設定

In [ ]:
# 必要なライブラリのインストール
!pip install -q unsloth wandb huggingface_hub datasets pyyaml

In [ ]:
import os
from google.colab import userdata

# ============================================================
# 環境変数設定（v7を参考に整理）
# ============================================================

# 1. HuggingFace設定
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["HF_REPO_ID"] = "kmd2525/qwen3-4b-sft-v9-clean-direct"
os.environ["HF_PRIVATE"] = "0"  # 公開リポジトリ

# 2. WandB設定
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["WANDB_PROJECT"] = "llm2025-sft"
os.environ["WANDB_RUN_NAME"] = "v9-clean-direct"

# 3. モデル・データセット関連
os.environ["SFT_BASE_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["SFT_DATASET_ID"] = "LOCAL:/content/merged_dataset_final_clean.jsonl"
os.environ["SFT_OUT_LORA_DIR"] = "/content/lora_structeval_qwen3_4b_v9"

# 4. 学習の基本パラメータ
os.environ["SFT_SEED"] = "3407"  # 再現性のため
os.environ["SFT_VAL_RATIO"] = "0.05"  # 検証データ割合（過学習監視用）
os.environ["SFT_MAX_SEQ_LEN"] = "1024"

# 5. LoRA (アダプタ) 設定 - Person R参考
os.environ["SFT_LORA_R"] = "64"
os.environ["SFT_LORA_ALPHA"] = "64"  # Person Rは r=alpha を推奨
os.environ["SFT_LORA_DROPOUT"] = "0.05"
os.environ["SFT_LORA_TARGET_MODULES"] = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"

# 6. ハイパーパラメータ - Person R参考
os.environ["SFT_EPOCHS"] = "2"
os.environ["SFT_PER_DEVICE_TRAIN_BS"] = "8"
os.environ["SFT_PER_DEVICE_EVAL_BS"] = "8"
os.environ["SFT_GRAD_ACCUM"] = "2"  # 実効バッチサイズ: 8*2=16
os.environ["SFT_LR"] = "5e-5"  # Person R推奨
os.environ["SFT_WARMUP_RATIO"] = "0.1"
os.environ["SFT_WEIGHT_DECAY"] = "0.05"  # 正則化

# 7. ステップ・保存設定
os.environ["SFT_MAX_STEPS"] = "-1"  # -1でエポックベース
os.environ["SFT_LOGGING_STEPS"] = "10"
os.environ["SFT_EVAL_STEPS"] = "50"
os.environ["SFT_SAVE_STEPS"] = "100"
os.environ["SFT_SAVE_TOTAL_LIMIT"] = "2"

# 注意: v9ではCoTマスク関連設定は不要（データが既にクリーン）
# os.environ["SFT_MASK_COT"] = "0"  # クリーンデータなので不要

print('v9 Clean Direct 環境変数設定完了')
print(f'  データセット: Person Y クリーンデータ (約1500件)')
print(f'  MAX_SEQ_LEN={os.environ["SFT_MAX_SEQ_LEN"]}, EPOCHS={os.environ["SFT_EPOCHS"]}')
print(f'  LR={os.environ["SFT_LR"]}, BS={os.environ["SFT_PER_DEVICE_TRAIN_BS"]}x{os.environ["SFT_GRAD_ACCUM"]}')
print(f'  LoRA: r={os.environ["SFT_LORA_R"]}, alpha={os.environ["SFT_LORA_ALPHA"]}')

## 1. データセットのアップロード

ローカルの `inputs/cleaned/merged_dataset_final_clean.jsonl` をColabにアップロードしてください。

In [ ]:
from google.colab import files

# ファイルをアップロード
print("merged_dataset_final_clean.jsonl をアップロードしてください")
uploaded = files.upload()

# アップロードされたファイル名を確認
DATASET_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_FILE}")

In [ ]:
# または、HuggingFaceにアップロード済みの場合
# DATASET_FILE = None  # この場合は下のセルでHFからロード
# os.environ["SFT_DATASET_ID"] = "kmd2525/v9-clean-dataset"

## 2. モデルとトークナイザーの読込み

In [ ]:
import torch
from unsloth import FastLanguageModel

def _getenv(key, default):
    return os.environ.get(key, default)

# 設定取得
BASE_MODEL = _getenv("SFT_BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
MAX_SEQ_LEN = int(_getenv("SFT_MAX_SEQ_LEN", "1024"))

print(f"Loading base model: {BASE_MODEL}")
print(f"Max sequence length: {MAX_SEQ_LEN}")

# モデル読込み
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

print("Model loaded successfully.")

## 3. LoRA設定

In [ ]:
# LoRA設定取得
LORA_R = int(_getenv("SFT_LORA_R", "64"))
LORA_ALPHA = int(_getenv("SFT_LORA_ALPHA", "64"))
LORA_DROPOUT = float(_getenv("SFT_LORA_DROPOUT", "0.05"))

print(f"LoRA r: {LORA_R}")
print(f"LoRA alpha: {LORA_ALPHA}")
print(f"LoRA dropout: {LORA_DROPOUT}")

# LoRA適用
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA configured.")

## 4. データセット読込み

In [ ]:
import json
from datasets import Dataset

# JSONLファイルから読込み
print(f"Loading dataset from: {DATASET_FILE}")

records = []
with open(DATASET_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded: {len(records)} records")

# フォーマット分布を確認
format_counts = {}
for item in records:
    fmt = item.get('format', 'unknown')
    format_counts[fmt] = format_counts.get(fmt, 0) + 1

print("\nFormat distribution:")
for fmt, count in sorted(format_counts.items()):
    print(f"  {fmt}: {count}")

# Datasetに変換
dataset = Dataset.from_list(records)

In [ ]:
# サンプルを確認
print("Sample data:")
print("="*60)
sample = records[0]
for msg in sample['messages']:
    role = msg['role']
    content = msg['content'][:300] + '...' if len(msg['content']) > 300 else msg['content']
    print(f"\n[{role}]")
    print(content)

## 5. データ前処理

In [ ]:
def format_chat(example):
    """チャット形式にフォーマット"""
    messages = example['messages']

    # tokenizer.apply_chat_templateを使用
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}

# データセットを変換
print("Formatting dataset...")
formatted_dataset = dataset.map(format_chat, remove_columns=dataset.column_names)

print(f"Formatted: {len(formatted_dataset)} samples")

# サンプル確認
print("\nFormatted sample:")
print(formatted_dataset[0]['text'][:500])

## 6. トレーナー設定

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import wandb

# 学習設定取得（v7を参考に整理）
SEED = int(_getenv("SFT_SEED", "3407"))
VAL_RATIO = float(_getenv("SFT_VAL_RATIO", "0.05"))
LR = float(_getenv("SFT_LR", "5e-5"))
EPOCHS = int(_getenv("SFT_EPOCHS", "2"))
BATCH_SIZE = int(_getenv("SFT_PER_DEVICE_TRAIN_BS", "8"))
EVAL_BATCH_SIZE = int(_getenv("SFT_PER_DEVICE_EVAL_BS", "8"))
GRAD_ACCUM = int(_getenv("SFT_GRAD_ACCUM", "2"))
WARMUP_RATIO = float(_getenv("SFT_WARMUP_RATIO", "0.1"))
WEIGHT_DECAY = float(_getenv("SFT_WEIGHT_DECAY", "0.05"))
LOGGING_STEPS = int(_getenv("SFT_LOGGING_STEPS", "10"))
EVAL_STEPS = int(_getenv("SFT_EVAL_STEPS", "50"))
SAVE_STEPS = int(_getenv("SFT_SAVE_STEPS", "100"))
SAVE_TOTAL_LIMIT = int(_getenv("SFT_SAVE_TOTAL_LIMIT", "2"))

# WandB設定
WANDB_PROJECT = _getenv("WANDB_PROJECT", "llm2025-sft")
WANDB_RUN_NAME = _getenv("WANDB_RUN_NAME", "v9-clean-direct")

print(f"Seed: {SEED}")
print(f"Validation ratio: {VAL_RATIO}")
print(f"Learning rate: {LR}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"Warmup ratio: {WARMUP_RATIO}")

# 検証データ分割
if VAL_RATIO > 0:
    split = formatted_dataset.train_test_split(test_size=VAL_RATIO, seed=SEED)
    train_dataset = split['train']
    eval_dataset = split['test']
    print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
else:
    train_dataset = formatted_dataset
    eval_dataset = None
    print(f"Train: {len(train_dataset)}, Eval: None")

# WandB初期化
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": BASE_MODEL,
        "dataset": "merged_dataset_final_clean.jsonl",
        "dataset_size": len(formatted_dataset),
        "train_size": len(train_dataset),
        "eval_size": len(eval_dataset) if eval_dataset else 0,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE * GRAD_ACCUM,
        "max_seq_len": MAX_SEQ_LEN,
        "strategy": "clean_direct",
        "seed": SEED,
    }
)

# 出力ディレクトリ
OUT_DIR = _getenv("SFT_OUT_LORA_DIR", "/content/lora_structeval_qwen3_4b_v9")
OUT_LORA_DIR = OUT_DIR

# TrainingArguments（v7を参考に整理）
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps" if eval_dataset else "no",
    eval_steps=EVAL_STEPS if eval_dataset else None,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True if eval_dataset else False,
    bf16=True,
    fp16=False,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="wandb",
    seed=SEED,
)

# Data Collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
)

# SFTTrainer（検証データ対応）
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    data_collator=data_collator,
    max_seq_length=MAX_SEQ_LEN,
)

print("Trainer configured.")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Eval samples: {len(eval_dataset) if eval_dataset else 0}")

## 7. 学習実行

In [ ]:
print("Starting training...")
print(f"Total samples: {len(formatted_dataset)}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Total steps: ~{len(formatted_dataset) * EPOCHS // BATCH_SIZE}")

# 学習実行
trainer.train()

print("Training completed!")

In [ ]:
# LoRA保存
print(f"Saving LoRA adapter to {OUT_LORA_DIR}")
model.save_pretrained(OUT_LORA_DIR)
tokenizer.save_pretrained(OUT_LORA_DIR)

# README作成
readme_content = f"""# Qwen3-4B SFT v9 Clean Direct

## 概要
- ベースモデル: {BASE_MODEL}
- データセット: merged_dataset_final_clean.jsonl ({len(records)} samples)
- 戦略: CoT除去済みクリーンデータ直接使用

## 設定
- LoRA r: {LORA_R}
- LoRA alpha: {LORA_ALPHA}
- Learning rate: {LR}
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}
- Max seq length: {MAX_SEQ_LEN}

## 特徴
- Person Y のデータクリーニングパイプラインで処理
- CoT（Approach/Output）完全削除
- 構文検証済みデータのみ使用
- Empty Think Injection なし

## フォーマット分布
""" + "\n".join([f"- {fmt}: {count}" for fmt, count in sorted(format_counts.items())])

with open(f"{OUT_LORA_DIR}/README.md", "w") as f:
    f.write(readme_content)

print("LoRA saved.")

## 8. HuggingFaceへアップロード

In [ ]:
from huggingface_hub import HfApi

HF_REPO_ID = _getenv("HF_REPO_ID", "kmd2525/qwen3-4b-sft-v9-clean-direct")
HF_TOKEN = _getenv("HF_TOKEN", "")
HF_PRIVATE = _getenv("HF_PRIVATE", "1") in ("1", "true", "True")

api = HfApi()

# リポジトリ作成
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=HF_PRIVATE,
    token=HF_TOKEN,
)

# LoRAアップロード
api.upload_folder(
    folder_path=OUT_LORA_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload v9 clean direct SFT LoRA adapter",
    token=HF_TOKEN,
)

print(f"LoRA uploaded to: https://huggingface.co/{HF_REPO_ID}")

## 9. LoRAマージとアップロード

In [ ]:
import shutil
from pathlib import Path
from peft import PeftModel

# マージモデル用リポジトリ
HF_MERGED_REPO_ID = HF_REPO_ID + "-merged"

STAGE_DIR = Path("/content/hf_upload_stage_merged")
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading base model for merge: {BASE_MODEL}")

# ベースモデル再読込み（フルプレシジョン）
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=False,  # マージ用にフルロード
)

print(f"Loading LoRA adapter from: {OUT_LORA_DIR}")
merged_model = PeftModel.from_pretrained(base_model, OUT_LORA_DIR)

print("Merging LoRA into base model...")
merged_model = merged_model.merge_and_unload()

print(f"Saving merged model to: {STAGE_DIR}")
merged_model.save_pretrained(str(STAGE_DIR), safe_serialization=True)
base_tokenizer.save_pretrained(str(STAGE_DIR))

# README コピー
shutil.copy(f"{OUT_LORA_DIR}/README.md", STAGE_DIR / "README.md")

print("Merged model saved.")

In [ ]:
# マージモデルをアップロード
api.create_repo(
    repo_id=HF_MERGED_REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=HF_PRIVATE,
    token=HF_TOKEN,
)

api.upload_folder(
    folder_path=str(STAGE_DIR),
    repo_id=HF_MERGED_REPO_ID,
    repo_type="model",
    commit_message="Upload v9 clean direct SFT merged model",
    token=HF_TOKEN,
)

print(f"Merged model uploaded to: https://huggingface.co/{HF_MERGED_REPO_ID}")

## 10. WandB終了

In [ ]:
wandb.finish()
print("WandB run finished.")

## 11: Colabインスタンスの削除（課金停止）

学習・アップロードがすべて完了したら、以下のセルを実行して **Colabランタイムを終了** してください。  
これにより、GPU課金が停止します（Colab Proの場合、ランタイムが生きている間は課金が続きます）。

⚠️ **注意**: 実行するとランタイムが即座に終了します。保存していないデータは失われます。


In [ ]:
# ============================================================
# Colabランタイムの削除（GPU課金停止）
# ============================================================
# 学習・アップロード完了後に実行してください。
# 実行するとランタイムが即座に終了し、GPUリソースが解放されます。
# Colab Pro/Pro+の場合、ランタイムが生きている限り課金が発生するため、
# 作業完了後は必ず実行することを推奨します。
from google.colab import runtime


try:
    print("[INFO] Colabランタイムを削除します...")
    runtime.unassign()
except Exception as e:
    print(f"[WARN] runtime.unassign() failed: {e}")
    print("[INFO] 手動でランタイムを終了してください:")
    print("       メニュー → ランタイム → ランタイムを接続解除して削除")

## まとめ

### 実験設定
- **データセット**: Person Y のクリーンデータセット (1500件)
- **Empty Think Injection**: なし
- **System role**: なし（データセットのまま）

### 次のステップ
1. 推論ノートブック（標準コード2）でマージモデルを使用して推論を実行
2. リーダーボードに提出してスコアを確認
3. 結果に応じて:
   - スコアが良ければ → 成功！
   - 改善余地があれば → Empty Think Injection を試す